# Lab 3

Finish a function in `lab3.py`, then run its cell. Every search returns a `Result`: the plan, plus what the search did to find it.

In [ ]:
%load_ext autoreload
%autoreload 2
import time
from lab2 import load, successors, goal_reached
from lab3 import *
from viz import explore, draw_search, workshop_problem

## A plan, and the search that found it

In [ ]:
domain, problem, actions = load("vacuum-domain.pddl", "vacuum-tiny.pddl")
r = bfs(problem, actions)
print(r)
r.plan

In [ ]:
s = problem.init
for a in r.plan:                      # replay it: every step must apply
    s = successors(s, actions)[a]
goal_reached(s, problem.goal)

`r.expanded` is the list of states the search took off its frontier, in order. Draw it.

In [ ]:
G = explore(problem.init, actions)
draw_search(bfs(problem, actions), G, problem.init, problem.goal, "vacuum")

In [ ]:
draw_search(dfs(problem, actions), G, problem.init, problem.goal, "vacuum")

Dark to light is the order of expansion; white states were never expanded.

Breadth-first fills the first room completely before it crosses a suck edge: everything one action away, then everything two away. Depth-first follows one branch as far as it goes and comes back with 7 actions where 5 would do. Same graph, same goal, same `successors`; only the frontier changed.

## Blocks: when luck looks like skill

In [ ]:
domain, problem, actions = load("blocks-domain.pddl", "blocks-sussman.pddl")
G = explore(problem.init, actions)
for name, fn in [("bfs", bfs), ("dfs", dfs)]:
    print(f"{name}: {fn(problem, actions)}")

In [ ]:
draw_search(bfs(problem, actions), G, problem.init, problem.goal, "blocks")

In [ ]:
draw_search(dfs(problem, actions), G, problem.init, problem.goal, "blocks")

Both found a 6-action plan, and depth-first did it in 10 expansions to breadth-first's 18. Look at the picture: it dived straight down the right branch, because of the order `successors` happens to return actions in. On the vacuum the same dive gave 7 actions for 5; below, on the workshop, it gives 27 for 9.

A search is judged by what it guarantees, not by a run that went well. Breadth-first guarantees the fewest actions. Depth-first guarantees a plan, and little memory.

## The workshop: fewest actions is not cheapest

In [ ]:
domain, problem, actions = load("domain.pddl", "problem.pddl")
results = {}
for name, fn in [("bfs", bfs), ("dfs", dfs), ("ucs", ucs)]:
    t = time.time(); results[name] = fn(problem, actions)
    print(f"{name}: {results[name]}  ({time.time() - t:.2f}s)")

In [ ]:
for a, b in zip(results["bfs"].plan, results["ucs"].plan):
    print(f"{action_cost(a)}  {a:36s} {action_cost(b)}  {b}")

Nine actions each, and the plans are different: breadth-first walked out and back (3 + 3) to deliver the dog before fetching the camera; uniform-cost did the camera first and walked once. Breadth-first stopped at the first goal state it generated, which is the shortest plan but not the cheapest. Uniform-cost kept going until nothing cheaper could be waiting, which costs it fewer expansions here (476 to 564) only by luck, and guarantees cost 12.

## Memory

In [ ]:
for name, r in results.items():
    print(f"{name}: frontier up to {r.max_frontier:4d} states, {len(r.expanded):4d} expanded")

Breadth-first and uniform-cost hold a whole wave of states at once. Depth-first holds one path and its siblings. That difference is why depth-first survives in worlds where the others run out of memory, and why iterative deepening exists.

## How far can blind search go?

In [ ]:
from lab2 import parse_problem, ground
for n in (1, 2, 3, 4, 5, 6):
    p = parse_problem(workshop_problem(n))
    acts = [g for a in domain.actions.values() for g in ground(a, p.objects, domain.types)]
    t = time.time(); r = bfs(p, acts)
    print(f"{n} items: {len(r.plan)} actions, {len(r.expanded):6d} expanded, frontier up to {r.max_frontier:5d}, {time.time() - t:5.2f}s")

Same nine-action plan every time, and five times more work per extra item. The plan did not get harder; the room got bigger, and blind search pays for every state within nine steps whether or not it matters.

Next week: search that looks where it is going.